# Module `algorithms/grasp.py`

GRASP = *Greedy Randomized Adaptive Search Procedure*. Chaque iteration :

1. **Construction gloutonne randomisee** via une *Restricted Candidate List* (RCL).
2. **Recherche locale** via `local_search` (relocate + swap + 2-opt).

Sur `max_iterations` redemarrages on retient la meilleure solution.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import grasp, greedy_randomized_construction, local_search, total_cost
from cesipath.solver_input import build_static_solver_input
import random


## 1. Instance de travail


In [2]:
cfg = GraphGenerationConfig(node_count=15, seed=3)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)
print(f"n = {instance.node_count}, capacite = {si.vehicle_capacity}, depot = {si.depot}")


n = 15, capacite = 40, depot = 0


## 2. Parametre cle : `rcl_alpha`

A chaque etape de la construction, les clients admissibles (demande <= capacite residuelle) sont tries par proximite au noeud courant. La RCL contient les `max(1, round(len(feasible) * rcl_alpha))` meilleurs. Le prochain client est tire **uniformement** dans la RCL.

- `rcl_alpha = 0` -> RCL de taille 1 -> construction **deterministe** (plus proche voisin pur).
- `rcl_alpha = 1` -> RCL = tous les admissibles -> construction **completement aleatoire**.
- Valeurs usuelles : 0.2 a 0.4 pour avoir de la diversite sans trop se degrader.


In [3]:
rng = random.Random(0)

for alpha in (0.0, 0.2, 0.5, 1.0):
    costs = []
    r = random.Random(0)
    for _ in range(20):
        routes = greedy_randomized_construction(si, alpha, r)
        costs.append(round(total_cost(routes, si.cost_matrix), 2))
    print(f"alpha={alpha:>3} : cout min={min(costs):>6.2f}  max={max(costs):>6.2f}  moy={sum(costs)/len(costs):.2f}")


alpha=0.0 : cout min=772.37  max=772.37  moy=772.37
alpha=0.2 : cout min=801.63  max=960.20  moy=865.50
alpha=0.5 : cout min=857.87  max=1101.68  moy=976.74
alpha=1.0 : cout min=957.08  max=1317.56  moy=1149.63


## 3. Construction puis polissage

Une seule construction ne donne pas une bonne solution. La recherche locale post-construction fait typiquement chuter le cout de 10 a 30 %.


In [4]:
rng = random.Random(7)
routes = greedy_randomized_construction(si, rcl_alpha=0.3, rng=rng)
before = total_cost(routes, si.cost_matrix)
polished = local_search(routes, si.cost_matrix, si.demands, si.vehicle_capacity)
after = total_cost(polished, si.cost_matrix)
print(f"construction  : {before:.2f}  ({len(routes)} tournees)")
print(f"local_search  : {after:.2f}  ({len(polished)} tournees)   gain={100*(before-after)/before:.1f} %")


construction  : 845.45  (3 tournees)
local_search  : 699.07  (3 tournees)   gain=17.3 %


## 4. GRASP complet

`grasp(solver_input, max_iterations, rcl_alpha, use_local_search, seed)` retourne une `VRPSolution` contenant la meilleure solution rencontree. Le flag `use_local_search=False` permet d'observer GRASP sans intensification (peu recommande en pratique).


In [5]:
for it in (5, 20, 50):
    sol = grasp(si, max_iterations=it, rcl_alpha=0.3, seed=42)
    print(f"max_iterations={it:>2} : cout={sol.total_cost:.2f}  routes={sol.route_count}")


max_iterations= 5 : cout=736.19  routes=3
max_iterations=20 : cout=699.07  routes=3
max_iterations=50 : cout=699.07  routes=3


## 5. Impact du `use_local_search`


In [6]:
sol_ls   = grasp(si, max_iterations=30, rcl_alpha=0.3, use_local_search=True,  seed=1)
sol_nols = grasp(si, max_iterations=30, rcl_alpha=0.3, use_local_search=False, seed=1)
print(f"avec    local_search : {sol_ls.total_cost:.2f}")
print(f"sans    local_search : {sol_nols.total_cost:.2f}")
print(f"amelioration relative: {100*(sol_nols.total_cost - sol_ls.total_cost)/sol_nols.total_cost:.1f} %")


avec    local_search : 699.07
sans    local_search : 826.61
amelioration relative: 15.4 %


## 6. Reproductibilite

Meme `seed` + memes parametres -> meme resultat exact. C'est fondamental pour debugger et pour les benchmarks.


In [7]:
a = grasp(si, max_iterations=30, rcl_alpha=0.3, seed=123)
b = grasp(si, max_iterations=30, rcl_alpha=0.3, seed=123)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True


## 7. Quand choisir GRASP ?

- **Budget faible a moyen** : quelques dizaines d'iterations suffisent.
- **Code simple a lire et a maintenir** : pas d'etat temporel (comme SA) ni de memoire (comme tabou).
- **Bon baseline** : GRASP est souvent le point de depart auquel comparer les autres algos.

Ses limites : pas de diversification globale (chaque redemarrage ignore l'historique), donc peut converger plusieurs fois vers le meme optimum local. Le tabou ou le genetique sont plus diversifiants.
